This notebook compares the performance in the training and test dose-response curves and pSTAT-dRNA correlations of the different IL-10 signaling mechanistic models (Control and Receptor memory (RAp)) in their Steady-State aproximations and ODE nuemrical simulations

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from scipy.optimize import curve_fit
from src.simulate_func import hill_func
import matplotlib.pyplot as plt

In [2]:
df_EC50_AMP = pd.read_csv('data/signaling/IL10_EC50_AMP.tsv.gz', sep="\t", compression="gzip")
# Get only data on certain variant and cells to fit
df_EC50_AMP_train = df_EC50_AMP.loc[(df_EC50_AMP["Variant"].isin(["WT","Super-10","R5A11D"]))&(df_EC50_AMP["Cell_type"].isin(["T8.Mean","T4.Mean","MO.Mean","ACH-000146","ACH-000786"]))]
df_EC50_AMP_train["Plot__Variant"] = df_EC50_AMP_train["Plot"].astype(str)+"__"+df_EC50_AMP_train["Variant"]
df_EC50_AMP_test = pd.concat([df_EC50_AMP.loc[(df_EC50_AMP["Variant"].isin(["WT","Super-10","R5A11D"]))&(~df_EC50_AMP["Cell_type"].isin(["T8.Mean","T4.Mean","MO.Mean","ACH-000146","ACH-000786"]))], pd.read_csv('data/signaling/IL10_EC50_AMP_CD8_Gorby.tsv.gz', sep="\t", compression="gzip")], ignore_index=True)
df_EC50_AMP_test["Plot__Variant"] = df_EC50_AMP_test["Plot"].astype(str)+"__"+df_EC50_AMP_test["Variant"]
df_EC50_AMP_test = df_EC50_AMP_test.loc[df_EC50_AMP_test["Cell_type"]!="ACH-002317"]

/tmp/ipykernel_4870/1623723409.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_EC50_AMP_train["Plot__Variant"] = df_EC50_AMP_train["Plot"].astype(str)+"__"+df_EC50_AMP_train["Variant"]


### Compare error between models (between topologies and ODE vs linear framework)

In [3]:
model_list = ["Control","Control_ODE","RAp","RAp_ODE"]
print("Error in training set:")
for model in model_list:
    df_sim = pd.read_csv("results/fit_param/simulations_fit_IL10_"+model+".csv")
    plot_num_old = 0
    # For each celltype/variant simulated
    for key in df_EC50_AMP_train["Plot__Variant"]:
        plot_num, variant = key.split("__")
        plot_num = int(plot_num)
        df_var = df_sim.loc[(df_sim["Plot"] == plot_num)&(df_sim["Variant"] == variant)]
        # Data is normalized to the first variant that appears in a plot (Determined from df_sim to be WT)
        if plot_num_old != plot_num:
            max_WT = df_var["Result"].max()
            fit, cov = curve_fit(hill_func, df_var["IL0"], df_var["Result"].values*100/(max_WT), bounds = ([0,df_var["IL0"].min()/10], [100, df_var["IL0"].max()*10]))
            df_EC50_AMP_train.loc[df_EC50_AMP_train["Plot__Variant"]==key,"Amp_m"] = fit[0]
            df_EC50_AMP_train.loc[df_EC50_AMP_train["Plot__Variant"]==key,"EC50_m"] = np.log10(fit[1])
        else:
            fit, cov = curve_fit(hill_func, df_var["IL0"], df_var["Result"].values*100/(max_WT), bounds = ([0,df_var["IL0"].min()/10], [df_var["Result"].max()*100/(max_WT)+5, df_var["IL0"].max()*10]))
            df_EC50_AMP_train.loc[df_EC50_AMP_train["Plot__Variant"]==key,"Amp_m"] = fit[0]
            df_EC50_AMP_train.loc[df_EC50_AMP_train["Plot__Variant"]==key,"EC50_m"] = np.log10(fit[1])
        plot_num_old = plot_num
    print("Model analyzed: "+model)
    print("MAE_EC50: "+str((df_EC50_AMP_train["EC50"]-df_EC50_AMP_train["EC50_m"]).abs().mean()))
    print("MAE_Amp: "+str((df_EC50_AMP_train["Amp"]-df_EC50_AMP_train["Amp_m"]).abs().mean()))

Error in training set:
Model analyzed: Control
MAE_EC50: 0.461895067467242
MAE_Amp: 10.721393480545286
Model analyzed: Control_ODE
MAE_EC50: 0.4602256623583663
MAE_Amp: 10.659518216194789
Model analyzed: RAp
MAE_EC50: 0.4713112531807475
MAE_Amp: 6.7340219998898245
Model analyzed: RAp_ODE
MAE_EC50: 0.26802998641677206
MAE_Amp: 4.951905779680054


In [4]:
model_list = ["Control","Control_ODE","RAp","RAp_ODE"]
print("Error in test set:")
for model in model_list:
    df_sim = pd.read_csv("results/fit_param/simulations_fit_IL10_"+model+".csv")
    plot_num_old = 0
    # For each celltype/variant simulated
    for key in df_EC50_AMP_test["Plot__Variant"]:
        plot_num, variant = key.split("__")
        plot_num = int(plot_num)
        df_var = df_sim.loc[(df_sim["Plot"] == plot_num)&(df_sim["Variant"] == variant)]
        # Data is normalized to the first variant that appears in a plot (Determined from df_sim to be WT)
        if plot_num_old != plot_num:
            max_WT = df_var["Result"].max()
            fit, cov = curve_fit(hill_func, df_var["IL0"], df_var["Result"].values*100/(max_WT), bounds = ([0,df_var["IL0"].min()/10], [100, df_var["IL0"].max()*10]))
            df_EC50_AMP_test.loc[df_EC50_AMP_test["Plot__Variant"]==key,"Amp_m"] = fit[0]
            df_EC50_AMP_test.loc[df_EC50_AMP_test["Plot__Variant"]==key,"EC50_m"] = np.log10(fit[1])
        else:
            fit, cov = curve_fit(hill_func, df_var["IL0"], df_var["Result"].values*100/(max_WT), bounds = ([0,df_var["IL0"].min()/10], [df_var["Result"].max()*100/(max_WT)+5, df_var["IL0"].max()*10]))
            df_EC50_AMP_test.loc[df_EC50_AMP_test["Plot__Variant"]==key,"Amp_m"] = fit[0]
            df_EC50_AMP_test.loc[df_EC50_AMP_test["Plot__Variant"]==key,"EC50_m"] = np.log10(fit[1])
        plot_num_old = plot_num
    print("Model analyzed: "+model)
    print("MAE_EC50: "+str((df_EC50_AMP_test["EC50"]-df_EC50_AMP_test["EC50_m"]).abs().mean()))
    print("MAE_Amp: "+str((df_EC50_AMP_test["Amp"]-df_EC50_AMP_test["Amp_m"]).abs().mean()))

Error in test set:
Model analyzed: Control
MAE_EC50: 0.24157823147208274
MAE_Amp: 9.374744175303011
Model analyzed: Control_ODE
MAE_EC50: 0.23684575616707715
MAE_Amp: 8.615197710446536
Model analyzed: RAp
MAE_EC50: 0.33404296349462453
MAE_Amp: 4.828572218565689
Model analyzed: RAp_ODE
MAE_EC50: 0.14887263831876965
MAE_Amp: 5.680497109526158


In [9]:
# Perfromance Receptor Memory model
df_EC50_AMP_train["EC50_e"] = (df_EC50_AMP_train["EC50"]-df_EC50_AMP_train["EC50_m"]).abs()
df_EC50_AMP_train["Amp_e"] = (df_EC50_AMP_train["Amp"]-df_EC50_AMP_train["Amp_m"]).abs()
df_EC50_AMP_test["EC50_e"] = (df_EC50_AMP_test["EC50"]-df_EC50_AMP_test["EC50_m"]).abs()
df_EC50_AMP_test["Amp_e"] = (df_EC50_AMP_test["Amp"]-df_EC50_AMP_test["Amp_m"]).abs()
df_EC50_AMP = pd.concat([df_EC50_AMP_train,df_EC50_AMP_test])
# Perfromance Control model
df_EC50_AMP_train_control["EC50_e"] = (df_EC50_AMP_train_control["EC50"]-df_EC50_AMP_train_control["EC50_m"]).abs()
df_EC50_AMP_train_control["Amp_e"] = (df_EC50_AMP_train_control["Amp"]-df_EC50_AMP_train_control["Amp_m"]).abs()
df_EC50_AMP_test_control["EC50_e"] = (df_EC50_AMP_test_control["EC50"]-df_EC50_AMP_test_control["EC50_m"]).abs()
df_EC50_AMP_test_control["Amp_e"] = (df_EC50_AMP_test_control["Amp"]-df_EC50_AMP_test_control["Amp_m"]).abs()
df_EC50_AMP_control = pd.concat([df_EC50_AMP_train_control,df_EC50_AMP_test_control])

### Is there a significant difference in the error between Control and Receptor Memory models?

In [46]:
# Is there a significant difference in the error between Control and Receptor Memory model performance
f_statistic, p_value = stats.ttest_ind(df_EC50_AMP["EC50_e"].dropna().values, df_EC50_AMP_control["EC50_e"].dropna().values, equal_var=False, alternative="less")
fig, ax = plt.subplots(1, 1, figsize=(5, 7), dpi=400)

ax.boxplot(
    [
        df_EC50_AMP_control["EC50_e"].dropna().values,
        df_EC50_AMP["EC50_e"].dropna().values
    ],
    widths=0.6,
    patch_artist=True,
    boxprops=dict(facecolor="grey", edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

# Labels
ax.set_ylabel("$MAE_{EC50}$", fontsize=20)
ax.set_xticklabels(["Control", "Receptor Memory"], fontsize=17)

plt.tight_layout()
plt.savefig(
    "/users/lserrano/qmarti/PhD_code/IL10/figures/fit_param/stat_tests/Plot_box_EC50_Control_RAp.pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.close(fig)
print("Control vs Receptor Memory EC50 p value: "+str(p_value))
print("N: "+str(len(df_EC50_AMP["EC50_e"].dropna().values)))

Control vs Receptor Memory EC50 p value: 0.01576768772151523
N: 24


In [45]:
# Is there a significant difference in the error between Control and Receptor Memory model performance
f_statistic, p_value = stats.ttest_ind(df_EC50_AMP.loc[df_EC50_AMP["Variant"]!="WT","Amp_e"].dropna().values, df_EC50_AMP_control.loc[df_EC50_AMP_control["Variant"]!="WT","Amp_e"].dropna().values, equal_var=False, alternative="less")
fig, ax = plt.subplots(1, 1, figsize=(5, 7), dpi=400)

ax.boxplot(
    [
        df_EC50_AMP_control.loc[df_EC50_AMP_control["Variant"]!="WT","Amp_e"].dropna().values,
        df_EC50_AMP.loc[df_EC50_AMP["Variant"]!="WT","Amp_e"].dropna().values
    ],
    widths=0.6,
    patch_artist=True,
    boxprops=dict(facecolor="grey", edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

# Labels
ax.set_ylabel("$MAE_{AMP}$", fontsize=20)
ax.set_xticklabels(["Control", "Receptor Memory"], fontsize=17)

plt.tight_layout()
plt.savefig(
    "/users/lserrano/qmarti/PhD_code/IL10/figures/fit_param/stat_tests/Plot_box_AMP_Control_RAp.pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.close(fig)
print("Control vs Receptor Memory AMP p value: "+str(p_value))
print("N: "+str(len(df_EC50_AMP.loc[df_EC50_AMP["Variant"]!="WT","Amp_e"].dropna().values)))

Control vs Receptor Memory AMP p value: 0.009602228434614455
N: 14


### Is there a significant difference in the error between train and test sets?

In [29]:
f_statistic, p_value = stats.ttest_ind(df_EC50_AMP_train["EC50_e"].dropna().values, df_EC50_AMP_test["EC50_e"].dropna().values, equal_var=False, alternative="less")
fig, ax = plt.subplots(1, 1, figsize=(5, 7), dpi=400)

ax.boxplot(
    [
        df_EC50_AMP_train["EC50_e"].dropna().to_list(),
        df_EC50_AMP_test["EC50_e"].dropna().to_list()
    ],
    widths=0.6,
    patch_artist=True,
    boxprops=dict(facecolor="grey", edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

# Labels
ax.set_ylabel("$MAE_{EC50}$", fontsize=20)
ax.set_xticklabels(["Training set", "Test set"], fontsize=17)

plt.tight_layout()
plt.savefig(
    "/users/lserrano/qmarti/PhD_code/IL10/figures/fit_param/stat_tests/Plot_box_EC50_train_test.pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.close(fig)
print("Train vs test EC50 p value: "+str(p_value))
print("N: "+str([len(df_EC50_AMP_train["EC50_e"].dropna().values),len(df_EC50_AMP_test["EC50_e"].dropna().values)]))

Train vs test EC50 p value: 0.9687440243200751
N: [14, 10]


In [30]:
# Is there a significant difference in the error between train and test sets
f_statistic, p_value = stats.ttest_ind(df_EC50_AMP_train["Amp_e"].dropna().values, df_EC50_AMP_test["Amp_e"].dropna().values, equal_var=False, alternative="less")
fig, ax = plt.subplots(1, 1, figsize=(5, 7), dpi=400)

ax.boxplot(
    [
        df_EC50_AMP_train["Amp_e"].dropna().to_list(),
        df_EC50_AMP_test["Amp_e"].dropna().to_list()
    ],
    widths=0.6,
    patch_artist=True,
    boxprops=dict(facecolor="grey", edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

# Labels
ax.set_ylabel("$MAE_{AMP}$", fontsize=20)
ax.set_xticklabels(["Training set", "Test set"], fontsize=17)

plt.tight_layout()
plt.savefig(
    "/users/lserrano/qmarti/PhD_code/IL10/figures/fit_param/stat_tests/Plot_box_AMP_train_test.pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.close(fig)
print("Train vs test AMP p value: "+str(p_value))
print("N: "+str([len(df_EC50_AMP_train["Amp_e"].dropna().values),len(df_EC50_AMP_test["Amp_e"].dropna().values)]))

Train vs test AMP p value: 0.4139662407940406
N: [18, 10]


### Is there a significant difference in the error between cell types?

In [49]:
cell_types_EC50 = [df_EC50_AMP.loc[df_EC50_AMP["Cell_type"]==cell,"EC50_e"].dropna().to_list() for cell in ['MO.Mean','T4.Mean','T8.Mean']]
f_statistic, p_value = stats.f_oneway(cell_types_EC50[0], cell_types_EC50[1], cell_types_EC50[2])
fig, ax = plt.subplots(1, 1, figsize=(7, 7), dpi=400)

ax.boxplot(
    [cell_types_EC50[0], cell_types_EC50[1], cell_types_EC50[2]],
    widths=0.6,
    patch_artist=True,
    boxprops=dict(facecolor="grey", edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

# Labels
ax.set_ylabel("$MAE_{EC50}$", fontsize=20)
ax.set_xticklabels(["Monocytes", "CD4⁺ T cells", "CD8⁺ T cells"], fontsize=17)

plt.tight_layout()
plt.savefig(
    "/users/lserrano/qmarti/PhD_code/IL10/figures/fit_param/stat_tests/Plot_box_EC50_cells.pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.close(fig)

print("Cell type EC50 p value: "+str(p_value))
print("N: "+str([len(cell_types_EC50[0]), len(cell_types_EC50[1]), len(cell_types_EC50[2])]))

Cell type EC50 p value: 0.25238000115768355
N: [6, 2, 6]


In [50]:
cell_types_AMP = [df_EC50_AMP.loc[df_EC50_AMP["Cell_type"]==cell,"Amp_e"].dropna().to_list() for cell in ['MO.Mean','T4.Mean','T8.Mean']]
f_statistic, p_value = stats.f_oneway(cell_types_AMP[0], cell_types_AMP[1], cell_types_AMP[2])
fig, ax = plt.subplots(1, 1, figsize=(7, 7), dpi=400)

ax.boxplot(
    [cell_types_AMP[0], cell_types_AMP[1], cell_types_AMP[2]],
    widths=0.6,
    patch_artist=True,
    boxprops=dict(facecolor="grey", edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

# Labels
ax.set_ylabel("$MAE_{AMP}$", fontsize=20)
ax.set_xticklabels(["Monocytes", "CD4⁺ T cells", "CD8⁺ T cells"], fontsize=17)

plt.tight_layout()
plt.savefig(
    "/users/lserrano/qmarti/PhD_code/IL10/figures/fit_param/stat_tests/Plot_box_AMP_cells.pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.close(fig)

print("Cell type EC50 p value: "+str(p_value))
print("N: "+str([len(cell_types_AMP[0]), len(cell_types_AMP[1]), len(cell_types_AMP[2])]))

Cell type EC50 p value: 0.8769998682352891
N: [6, 4, 8]


### Is there a significant difference in the error between IL-10 variants?

In [51]:
variant_EC50 = [df_EC50_AMP.loc[df_EC50_AMP["Variant"]==var,"EC50_e"].dropna().to_list() for var in ['WT','R5A11D','Super-10']]
f_statistic, p_value = stats.f_oneway(variant_EC50[0], variant_EC50[1], variant_EC50[2])
fig, ax = plt.subplots(1, 1, figsize=(7, 7), dpi=400)

ax.boxplot(
    [variant_EC50[0], variant_EC50[1], variant_EC50[2]],
    widths=0.6,
    patch_artist=True,
    boxprops=dict(facecolor="grey", edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

# Labels
ax.set_ylabel("$MAE_{EC50}$", fontsize=20)
ax.set_xticklabels(['WT IL-10','R5A11D','Super-10'], fontsize=17)

plt.tight_layout()
plt.savefig(
    "/users/lserrano/qmarti/PhD_code/IL10/figures/fit_param/stat_tests/Plot_box_EC50_variants.pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.close(fig)
print("Variant EC50 p value: "+str(p_value))
print("N: "+str([len(variant_EC50[0]), len(variant_EC50[1]), len(variant_EC50[2])]))

Variant EC50 p value: 0.8710172472282645
N: [12, 4, 8]


In [52]:
variant_AMP = [df_EC50_AMP.loc[df_EC50_AMP["Variant"]==var,"Amp_e"].dropna().to_list() for var in ['R5A11D','Super-10']]
f_statistic, p_value = stats.f_oneway(variant_AMP[0], variant_AMP[1])
fig, ax = plt.subplots(1, 1, figsize=(5, 7), dpi=400)

ax.boxplot(
    [variant_AMP[0], variant_AMP[1]],
    widths=0.6,
    patch_artist=True,
    boxprops=dict(facecolor="grey", edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

# Labels
ax.set_ylabel("$MAE_{AMP}$", fontsize=20)
ax.set_xticklabels(['R5A11D','Super-10'], fontsize=17)

plt.tight_layout()
plt.savefig(
    "/users/lserrano/qmarti/PhD_code/IL10/figures/fit_param/stat_tests/Plot_box_AMP_variants.pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.close(fig)

print("Variant AMP p value: "+str(p_value))
print("N: "+str([len(variant_AMP[0]), len(variant_AMP[1])]))

Variant AMP p value: 0.2770339252337045
N: [4, 10]


### Is there a significant difference in the error between cells with infered or experimentally measured Receptors/STAT?

In [25]:
df_IC_data = pd.read_csv('data/expression/whole_dataset_IL10.tsv.gz', sep="\t", compression="gzip")
df_IC_data.loc[df_IC_data["Prot. count dataset num."]!=0,"Prot. count dataset num."]=1
df_measured = pd.DataFrame(df_IC_data.loc[df_IC_data["Cell"].isin(df_EC50_AMP["Cell_type"].drop_duplicates())].groupby("Cell")["Prot. count dataset num."].sum())
# Separate into cells with all components measured or some/all estimated via linear models
df_EC50_AMP_measured = df_EC50_AMP.loc[df_EC50_AMP["Cell_type"].isin(df_measured.loc[df_measured["Prot. count dataset num."]==4].index)]
df_EC50_AMP_estimated = df_EC50_AMP.loc[df_EC50_AMP["Cell_type"].isin(df_measured.loc[df_measured["Prot. count dataset num."]!=4].index)]

In [32]:
f_statistic, p_value = stats.ttest_ind(df_EC50_AMP_measured["EC50_e"].dropna().values, df_EC50_AMP_estimated["EC50_e"].dropna().values, equal_var=False, alternative="less")
fig, ax = plt.subplots(1, 1, figsize=(5, 7), dpi=400)

ax.boxplot(
    [
        df_EC50_AMP_measured["EC50_e"].dropna().to_list(),
        df_EC50_AMP_estimated["EC50_e"].dropna().to_list()
    ],
    widths=0.6,
    patch_artist=True,
    boxprops=dict(facecolor="grey", edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

# Labels
ax.set_ylabel("$MAE_{EC50}$", fontsize=20)
ax.set_xticklabels(["Measured IC", "Estimated IC"], fontsize=17)

plt.tight_layout()
plt.savefig(
    "/users/lserrano/qmarti/PhD_code/IL10/figures/fit_param/stat_tests/Plot_box_EC50_measured_estimated.pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.close(fig)
print("IC EC50 p value: "+str(p_value))
print("N: "+str([len(df_EC50_AMP_measured["EC50_e"].dropna().values),len(df_EC50_AMP_estimated["EC50_e"].dropna().values)]))

IC EC50 p value: 0.36734790177927656
N: [16, 8]


In [31]:
# Is there a significant difference in the error between train and test sets
f_statistic, p_value = stats.ttest_ind(df_EC50_AMP_measured["Amp_e"].dropna().values, df_EC50_AMP_estimated["Amp_e"].dropna().values, equal_var=False, alternative="less")
fig, ax = plt.subplots(1, 1, figsize=(5, 7), dpi=400)

ax.boxplot(
    [
        df_EC50_AMP_measured["Amp_e"].dropna().to_list(),
        df_EC50_AMP_estimated["Amp_e"].dropna().to_list()
    ],
    widths=0.6,
    patch_artist=True,
    boxprops=dict(facecolor="grey", edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

# Labels
ax.set_ylabel("$MAE_{AMP}$", fontsize=20)
ax.set_xticklabels(["Measured IC", "Estimated IC"], fontsize=17)

plt.tight_layout()
plt.savefig(
    "/users/lserrano/qmarti/PhD_code/IL10/figures/fit_param/stat_tests/Plot_box_AMP_measured_estimated.pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.close(fig)
print("IC AMP p value: "+str(p_value))
print("N: "+str([len(df_EC50_AMP_measured["Amp_e"].dropna().values),len(df_EC50_AMP_estimated["Amp_e"].dropna().values)]))

IC AMP p value: 0.7510810628697869
N: [20, 8]
